# Combined Inference Overlay

Ce notebook utilise le module `bcs_pipeline.inference` pour exécuter les **trois pipelines** sur une même image et superposer leurs sorties :

1. **Classification de race** — ResNet50 (Stanford Dogs, 120 races)
2. **Segmentation sémantique** — au choix : **DeepLabV3-ResNet50** (fine-tuned sur Oxford Pet) ou **SAM 2** (zero-shot foundation)
3. **Détection de pose** — YOLOv8 custom (bounding boxes + keypoints)

Datasets testés : **Reddit** (out-of-distribution), **Stanford Dogs** (in-distribution classifieur), **Oxford-IIIT Pet** (in-distribution segmentation, contient chiens + chats).

Un widget interactif en fin de notebook permet d'explorer les résultats par dataset, race et image.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

print(f"Repo root: {REPO_ROOT}")

In [ ]:
# Checkpoints
CLASSIFICATION_CKPT = REPO_ROOT / "experiments/resnet50_adam_cosine_annealing/checkpoints/epoch=epoch=15-val_acc=val_acc=0.79-step=8240.ckpt"
DEEPLAB_CKPT        = REPO_ROOT / "experiments/deeplabv3_resnet50_adam_cosine_annealing/checkpoints/last-v1.ckpt"
SAM2_CKPT           = REPO_ROOT / "checkpoints/sam2.1_hiera_large.pt"
POSE_CKPT           = REPO_ROOT / "runs/pose/train/weights/best.pt"

# Dataset roots
STANFORD_ROOT = REPO_ROOT / "data/stanford_dogs/images"               # contains an Images/ subfolder
STANFORD_IMAGES = STANFORD_ROOT / "Images"
OXFORD_IMAGES = REPO_ROOT / "data/Oxford-IIIT_pet_dataset/images"
REDDIT_DIR    = REPO_ROOT / "data/Reddit_example"

OUTPUT_DIR = REPO_ROOT / "outputs" / "notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for path in [CLASSIFICATION_CKPT, DEEPLAB_CKPT, SAM2_CKPT, POSE_CKPT, STANFORD_IMAGES, OXFORD_IMAGES, REDDIT_DIR]:
    assert path.exists(), f"Missing: {path}"
print("All paths exist.")

## 2. Helpers factorisés

On charge les modèles **une seule fois** (cache global) et on définit deux helpers :

- `infer_all(image_path, seg_backend=..., sam2_mode=...)` — exécute les 3 pipelines.
- `show_inference(image_path, seg_backend=..., sam2_mode=..., title=...)` — appelle `infer_all`, puis affiche **source à gauche, overlay à droite** sur la même ligne.

Le backend de segmentation se choisit via `seg_backend="deeplab"` (par défaut) ou `seg_backend="sam2"`. Pour SAM 2, trois modes de prompting :

- `prompted` (défaut) : un point positif au centre de l'image
- `automatic` : grille dense de points générés (plus lent, pour sujets décentrés)
- `pose_prompted` : utilise la **bounding box et les keypoints YOLO** comme prompts SAM 2 — la box contraint la zone, les keypoints servent de points positifs

En mode `pose_prompted`, la pose est exécutée **avant** la segmentation pour fournir le prompt.

In [ ]:
from bcs_pipeline.inference import (
    load_classification_model, load_class_names, predict_single,
    load_segmentation_backend, predict_segmentation_with,
    load_pose_model, predict_pose,
    render_combined,
)

# Lazy cache: backends loaded on first use.
_MODELS: dict = {
    "cls": None,
    "class_names": None,
    "seg": {},        # keyed by backend name
    "pose": None,
}


def _ensure_classifier():
    if _MODELS["cls"] is None:
        _MODELS["cls"] = load_classification_model(str(CLASSIFICATION_CKPT))
        _MODELS["class_names"] = load_class_names(str(STANFORD_ROOT))
    return _MODELS["cls"], _MODELS["class_names"]


def _ensure_segmenter(backend: str):
    if backend not in _MODELS["seg"]:
        ckpt = str(SAM2_CKPT) if backend == "sam2" else str(DEEPLAB_CKPT)
        _MODELS["seg"][backend] = load_segmentation_backend(backend, ckpt)
    return _MODELS["seg"][backend]


def _ensure_pose():
    if _MODELS["pose"] is None:
        _MODELS["pose"] = load_pose_model(str(POSE_CKPT))
    return _MODELS["pose"]


def infer_all(
    image_path,
    seg_backend: str = "deeplab",
    sam2_mode: str = "prompted",
    top_k: int = 5,
    conf_threshold: float = 0.25,
) -> dict:
    """Run the three pipelines on a single image and return all outputs.

    When ``seg_backend == "sam2"`` and ``sam2_mode == "pose_prompted"``, the
    pose detection runs first so its outputs can be fed as SAM 2 prompts.
    """
    img = Image.open(str(image_path)).convert("RGB")
    cls_model, class_names = _ensure_classifier()
    seg_handle = _ensure_segmenter(seg_backend)
    pose_model = _ensure_pose()

    cls = predict_single(cls_model, img, class_names=class_names, top_k=top_k)

    needs_pose_first = seg_backend == "sam2" and sam2_mode == "pose_prompted"
    pose = predict_pose(pose_model, img, conf_threshold=conf_threshold) if needs_pose_first else None

    seg = predict_segmentation_with(
        seg_backend, seg_handle, img,
        sam2_mode=sam2_mode, pose_result=pose,
    )

    if pose is None:
        pose = predict_pose(pose_model, img, conf_threshold=conf_threshold)

    return {"image": img, "classification": cls, "segmentation": seg, "pose": pose}


def show_inference(
    image_path,
    seg_backend: str = "deeplab",
    sam2_mode: str = "prompted",
    title: str | None = None,
    figsize=(12, 6),
) -> dict:
    """Run inference and display source + overlay side by side."""
    res = infer_all(image_path, seg_backend=seg_backend, sam2_mode=sam2_mode)
    composed = render_combined(
        res["image"],
        classification=res["classification"],
        segmentation=res["segmentation"],
        pose=res["pose"],
    )
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    axes[0].imshow(res["image"])
    axes[0].set_title("Source"); axes[0].axis("off")
    axes[1].imshow(composed)
    cls = res["classification"]
    backend_label = res["segmentation"].get("backend", seg_backend)
    axes[1].set_title(
        f"Overlay [{backend_label}] — {cls['class_name']} ({cls['confidence']*100:.1f}%) | pose: {res['pose']['num_detections']} det"
    )
    axes[1].axis("off")
    if title:
        fig.suptitle(title, fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()
    return res

## 3. Comparaison DeepLabV3 vs SAM 2 (3 modes)

DeepLabV3 a été fine-tuné sur Oxford Pet (3 classes : foreground / background / border).
SAM 2 est zero-shot et produit des contours bien plus nets. Les 3 modes SAM 2 :

- **prompted** : 1 point au centre — rapide, marche si le sujet est centré.
- **automatic** : grille de points, sélection du masque le plus central — robuste aux sujets décentrés.
- **pose_prompted** : la **bbox YOLO contraint la zone** et les **keypoints** servent de prompts positifs — utile quand la pose détecte bien le chien (typiquement la meilleure qualité).

In [ ]:
TEST_IMAGE = REDDIT_DIR / "is-my-dog-overweight-v0-am4q7ltvecng1.webp"
show_inference(TEST_IMAGE, seg_backend="deeplab", title="DeepLabV3 (fine-tuned)")
show_inference(TEST_IMAGE, seg_backend="sam2",    sam2_mode="prompted",      title="SAM 2 — prompted (point central)")
show_inference(TEST_IMAGE, seg_backend="sam2",    sam2_mode="pose_prompted", title="SAM 2 — pose_prompted (bbox + keypoints YOLO)")

## 4. Reddit (out-of-distribution)

Photos prises au hasard sur Reddit — utile pour évaluer la robustesse en conditions réelles.

In [ ]:
for img_path in sorted(REDDIT_DIR.glob("*.webp")):
    show_inference(img_path, seg_backend="sam2", title=f"Reddit — {img_path.name}")

## 5. Stanford Dogs (in-distribution pour le classifieur)

In [ ]:
STANFORD_BREEDS = [
    "n02085620-Chihuahua",
    "n02099601-golden_retriever",
    "n02105641-Old_English_sheepdog",
    "n02110958-pug",
]

stanford_samples = []
for breed_dir_name in STANFORD_BREEDS:
    breed_dir = STANFORD_IMAGES / breed_dir_name
    if not breed_dir.exists():
        candidates = sorted(STANFORD_IMAGES.glob(f"{breed_dir_name.split('-')[0]}*"))
        if not candidates:
            print(f"Skipping (not found): {breed_dir_name}")
            continue
        breed_dir = candidates[0]
    sample = next(iter(sorted(breed_dir.glob("*.jpg"))), None)
    if sample is not None:
        stanford_samples.append((breed_dir.name, sample))

for breed_label, sample in stanford_samples:
    expected = breed_label.split("-", 1)[1] if "-" in breed_label else breed_label
    show_inference(sample, seg_backend="sam2", title=f"Stanford — attendu: {expected}")

## 6. Oxford-IIIT Pet (chiens + chats)

Le classifieur n'a vu que des chiens — il retournera n'importe quoi sur un chat. La segmentation marche très bien dans les deux cas (SAM 2 est zero-shot, DeepLabV3 a été entraîné sur ces images).

In [ ]:
OXFORD_SAMPLES_PREFIX = [
    "american_bulldog_100",   # chien
    "basset_hound_100",       # chien
    "Bengal_100",             # chat
    "Birman_100",             # chat
]

oxford_samples = []
for prefix in OXFORD_SAMPLES_PREFIX:
    candidates = sorted(OXFORD_IMAGES.glob(f"{prefix}.jpg"))
    if candidates:
        oxford_samples.append((prefix, candidates[0]))

for label, sample in oxford_samples:
    show_inference(sample, seg_backend="sam2", title=f"Oxford — {label}")

## 7. Widget interactif

Sélectionne **dataset → race/sous-dossier → image**, choisis le **backend de segmentation**, et clique sur **Lancer l'inférence**. Le widget renvoie l'image source à gauche, l'overlay à droite, et un résumé textuel des observations (top-5 races, distribution des classes seg, nombre de détections pose).

Utile pour rapidement comparer le comportement des 3 pipelines sur n'importe quelle image disponible localement.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output


def _list_image_files(folder: Path, exts=(".jpg", ".jpeg", ".png", ".webp")) -> list[Path]:
    if not folder.is_dir():
        return []
    return sorted(p for p in folder.iterdir() if p.suffix.lower() in exts and p.is_file())


def _stanford_groups() -> dict[str, list[Path]]:
    """{breed_name: [image_paths]} for all 120 Stanford breeds."""
    groups = {}
    for breed_dir in sorted(p for p in STANFORD_IMAGES.iterdir() if p.is_dir()):
        breed_name = breed_dir.name.split("-", 1)[1] if "-" in breed_dir.name else breed_dir.name
        groups[breed_name] = _list_image_files(breed_dir)
    return groups


def _oxford_groups() -> dict[str, list[Path]]:
    """{breed_prefix: [image_paths]} grouping Oxford filenames by their prefix."""
    groups: dict[str, list[Path]] = {}
    for img_path in _list_image_files(OXFORD_IMAGES):
        prefix = "_".join(img_path.stem.split("_")[:-1])  # drop trailing index
        groups.setdefault(prefix, []).append(img_path)
    return {k: sorted(v) for k, v in sorted(groups.items())}


def _reddit_groups() -> dict[str, list[Path]]:
    files = _list_image_files(REDDIT_DIR)
    return {"all": files} if files else {}


DATASETS = {
    "Reddit": _reddit_groups(),
    "Stanford Dogs": _stanford_groups(),
    "Oxford-IIIT Pet": _oxford_groups(),
}

print("Available datasets:")
for name, groups in DATASETS.items():
    n_images = sum(len(v) for v in groups.values())
    print(f"  {name:<20s} | {len(groups):>4d} groups | {n_images:>5d} images")

In [ ]:
dataset_dd = widgets.Dropdown(
    options=list(DATASETS.keys()), value="Reddit",
    description="Dataset:", layout=widgets.Layout(width="320px"),
)
breed_dd = widgets.Dropdown(
    options=[], description="Race / groupe:",
    layout=widgets.Layout(width="380px"),
    style={"description_width": "100px"},
)
image_dd = widgets.Dropdown(
    options=[], description="Image:",
    layout=widgets.Layout(width="380px"),
    style={"description_width": "60px"},
)
backend_dd = widgets.Dropdown(
    options=[("DeepLabV3 (fine-tuned)", "deeplab"), ("SAM 2 (zero-shot)", "sam2")],
    value="sam2", description="Segmentation:",
    layout=widgets.Layout(width="320px"),
    style={"description_width": "100px"},
)
sam2_mode_dd = widgets.Dropdown(
    options=[
        ("prompted (centre)", "prompted"),
        ("automatic (grille)", "automatic"),
        ("pose_prompted (bbox + keypoints YOLO)", "pose_prompted"),
    ],
    value="pose_prompted", description="SAM 2 mode:",
    layout=widgets.Layout(width="380px"),
    style={"description_width": "100px"},
)
run_btn = widgets.Button(description="Lancer l'inférence", button_style="primary",
                         icon="play", layout=widgets.Layout(width="200px"))
out = widgets.Output()


def _on_dataset_change(change=None):
    groups = DATASETS[dataset_dd.value]
    breed_dd.options = list(groups.keys())
    if breed_dd.options:
        breed_dd.value = breed_dd.options[0]
    _on_breed_change()


def _on_breed_change(change=None):
    groups = DATASETS[dataset_dd.value]
    files = groups.get(breed_dd.value, [])
    image_dd.options = [(p.name, p) for p in files]
    if image_dd.options:
        image_dd.value = image_dd.options[0][1]


def _on_backend_change(change=None):
    # Disable SAM2 mode dropdown when DeepLab is selected.
    sam2_mode_dd.disabled = backend_dd.value != "sam2"


def _on_run(_btn):
    with out:
        clear_output(wait=True)
        if not image_dd.value:
            print("Aucune image sélectionnée.")
            return
        res = infer_all(
            image_dd.value,
            seg_backend=backend_dd.value,
            sam2_mode=sam2_mode_dd.value,
        )
        composed = render_combined(
            res["image"],
            classification=res["classification"],
            segmentation=res["segmentation"],
            pose=res["pose"],
        )
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))
        axes[0].imshow(res["image"]); axes[0].set_title("Source"); axes[0].axis("off")
        axes[1].imshow(composed)
        cls = res["classification"]
        backend_label = res["segmentation"].get("backend", backend_dd.value)
        axes[1].set_title(
            f"Overlay [{backend_label}] — {cls['class_name']} ({cls['confidence']*100:.1f}%)"
        )
        axes[1].axis("off")
        plt.tight_layout(); plt.show()

        # Textual observations
        print(f"Image       : {image_dd.value.name}  ({res['image'].size[0]}×{res['image'].size[1]})")
        print(f"Top-5 races :")
        for i, e in enumerate(cls["top_k"], 1):
            print(f"   {i}. {e['class_name']:<30s} {e['confidence']*100:6.2f}%")
        mask = res["segmentation"]["mask"]
        unique, counts = np.unique(mask, return_counts=True)
        dist = "  ".join(f"cls{int(u)}={c/mask.size*100:.1f}%" for u, c in zip(unique, counts))
        print(f"Segmentation: backend={backend_label} | {dist}")
        pose = res["pose"]
        print(f"Pose        : {pose['num_detections']} détection(s)"
              + (f" | conf={pose['box_confs'][0]:.2f}" if pose['num_detections'] > 0 else ""))


dataset_dd.observe(_on_dataset_change, names="value")
breed_dd.observe(_on_breed_change, names="value")
backend_dd.observe(_on_backend_change, names="value")
run_btn.on_click(_on_run)
_on_dataset_change()
_on_backend_change()

ui = widgets.VBox([
    widgets.HBox([dataset_dd, backend_dd]),
    widgets.HBox([breed_dd, image_dd]),
    sam2_mode_dd,
    run_btn,
    out,
])
display(ui)

## 8. Sauvegarde sur disque (CLI-style)

In [ ]:
from bcs_pipeline.inference import run_full_inference

result = run_full_inference(
    image=str(REDDIT_DIR / "is-my-dog-overweight-v0-am4q7ltvecng1.webp"),
    classification_ckpt=str(CLASSIFICATION_CKPT),
    segmentation_ckpt=str(SAM2_CKPT),
    segmentation_backend="sam2",
    sam2_mode="prompted",
    pose_ckpt=str(POSE_CKPT),
    output_path=str(OUTPUT_DIR / "oneshot_sam2.png"),
    data_dir=str(STANFORD_ROOT),
)
print(f"Saved: {result['output_path']}")
print(f"Breed: {result['classification']['class_name']} ({result['classification']['confidence']*100:.2f}%)")
print(f"Seg backend: {result['segmentation']['backend']}")

## 9. API publique

In [ ]:
import bcs_pipeline.inference as bi
for name in sorted(bi.__all__):
    obj = getattr(bi, name)
    kind = "function" if callable(obj) else type(obj).__name__
    print(f"  {name:<35s} ({kind})")